# Chest X-ray pneumonia classifier: ResNet18 transfer learning

Same data as `04_chest_Xray_CNN.ipynb`, but the hand-built CNN is replaced by an **ImageNet-pretrained ResNet18**.

* The backbone is **frozen**: only the new final layer (`Linear(512, 1)` + sigmoid) is trained.
* Images are read as 3-channel RGB and normalised with the ImageNet mean/std the backbone was trained with.
* Training and validation use a reproducible, class-stratified 80/20 split of `train/`. Only validation drives early stopping; `test/` is evaluated after model selection.
* This is an image-level split: patient IDs are unavailable, so patient independence is unverified. Earlier versions used `test/` for model selection; its historical exposure cannot be undone by this fix. A fresh external set is needed for an untouched evaluation.
* Restart the kernel and run all cells to train a fresh model; existing checkpoints and metrics predate this fix.

At the end, the trained model is saved to `resnet18_pneumonia_model.pt`, with its metadata also written to `resnet18_pneumonia_model.json`.

In [ ]:
## On Google Colab, run the following lines before the imports:
#!pip install pytorch-model-summary
#!wget https://github.com/Bjarten/early-stopping-pytorch/raw/refs/heads/main/early_stopping_pytorch/early_stopping.py
#!mv early_stopping.py pytorchtools.py


In [ ]:
## On Google Colab, run the following lines to download the sample images and datasets:

## Downloading 2 images:
#!mkdir -p images
#!wget https://github.com/sib-swiss/pytorch-practical-training/raw/refs/heads/master/images/NORMAL-1003233-0001.jpeg -O images/NORMAL-1003233-0001.jpeg
#!wget https://github.com/sib-swiss/pytorch-practical-training/raw/refs/heads/master/images/BACTERIA-1008087-0001.jpeg -O images/BACTERIA-1008087-0001.jpeg

## Downloading and unzipping the resized datasets:
#!wget 'https://sibcloud-my.sharepoint.com/:u:/g/personal/wandrille_duchemin_sib_swiss/ESDXFrlw6JJGiK8X7xG4aVEB06cxW82KyK_KWXMwccIVhw?download=1' -O chest_xray_224.zip
#!wget 'https://sibcloud-my.sharepoint.com/:u:/g/personal/wandrille_duchemin_sib_swiss/EZLYtPxO4dlPr-2kysgQFYQBA0iLG3fEWnwrnlwvoHbZwg?download=1' -O chest_xray_64.zip

#!mkdir -p data
#!unzip -o chest_xray_224.zip
#!mv -n chest_xray_224 data/

#!unzip -o chest_xray_64.zip
#!mv -n chest_xray_64 data/


In [ ]:
import json
import random
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

import torchvision
from torchvision.transforms import v2
from torchvision.io import read_image, ImageReadMode
from torchvision.models import resnet18, ResNet18_Weights

from pytorchtools import EarlyStopping

# Get cpu, gpu or mps device for training.
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

SEED = 0
torch.manual_seed(SEED)

In [ ]:
def train(dataloader, model, loss_fn, optimizer ,  echo = True , echo_batch = False):
    
    size = len(dataloader.dataset) # how many batches do we have
    model.train() #     Sets the module in training mode.
    
    for batch, (X,y) in enumerate(dataloader): # for each batch
        X = X.to(device) # send the data to the GPU or whatever device you use for training
        y = y.to(device) # send the data to the GPU or whatever device you use for training

        # Compute prediction error
        pred = model(X)              # prediction for the model -> forward pass
        loss = loss_fn(pred, y)      # loss function from these prediction        
        
        # Backpropagation
        loss.backward()              # backward propagation 
        #                            https://ml-cheatsheet.readthedocs.io/en/latest/backpropagation.html
        #                            https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html
        
        optimizer.step()             
        optimizer.zero_grad()        # reset the gradients
                                     # https://stackoverflow.com/questions/48001598/why-do-we-need-to-call-zero-grad-in-pytorch

        if echo_batch:
            current =  (batch) * dataloader.batch_size +  len(X)
            print(f"Train loss: {loss.item():>7f}  [{current:>5d}/{size:>5d}]")
    
    if echo:
        print(f"Train loss: {loss.item():>7f}")

    # return the last batch loss
    return loss.item()

def valid(dataloader, model, loss_fn, echo = True):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval() #     Sets the module in evaluation mode
    valid_loss = 0
    with torch.no_grad(): ## disables tracking of gradient: prevent accidental training + speeds up computation
        for X,y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            valid_loss += loss_fn(pred, y).item() * len(y)  ## accumulating the loss function over the batches
            
    valid_loss /= size

    if echo:
        print(f"Valid Error: {valid_loss:>8f}")
    ## return the average loss / batch
    return valid_loss

def get_model_accuracy(model, dataloader):
    # no_grad alone does not prevent BatchNorm running-statistic updates.
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            correct += ((pred.reshape(-1) > 0.5) == y.reshape(-1)).sum().item()
            total += y.numel()
    return correct / total


In [ ]:
WEIGHTS = ResNet18_Weights.IMAGENET1K_V1
IMAGENET_MEAN = WEIGHTS.transforms().mean
IMAGENET_STD = WEIGHTS.transforms().std

# images are already 224x224: skip the weights' default Resize(256)+CenterCrop(224), which would cut the edges
rn_transform = v2.Compose([
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
read_image_rgb = lambda path: read_image(path, mode=ImageReadMode.RGB)

TRAIN_DIR = 'data/chest_xray_224/train'
TEST_DIR = 'data/chest_xray_224/test'
VALID_FRACTION = 0.2

full_train_dataset = torchvision.datasets.ImageFolder(
    TRAIN_DIR, loader=read_image_rgb, transform=rn_transform,
    target_transform=lambda x: torch.tensor([x], dtype=torch.float32))

# Split only the training folder, independently within each class.
# Local RNG keeps membership independent of model initialization and cell reruns.
rng = random.Random(SEED)
train_indices, valid_indices = [], []
for label in sorted(set(full_train_dataset.targets)):
    indices = [i for i, target in enumerate(full_train_dataset.targets) if target == label]
    rng.shuffle(indices)
    n_valid = round(len(indices) * VALID_FRACTION)
    valid_indices.extend(indices[:n_valid])
    train_indices.extend(indices[n_valid:])
train_indices.sort()
valid_indices.sort()
assert set(train_indices).isdisjoint(valid_indices)
assert sorted(train_indices + valid_indices) == list(range(len(full_train_dataset)))
train_dataset = Subset(full_train_dataset, train_indices)
valid_dataset = Subset(full_train_dataset, valid_indices)

batch_size = 64
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
valid_dataloader = DataLoader(valid_dataset, shuffle=False, batch_size=batch_size)

print(full_train_dataset.class_to_idx, len(train_dataset), len(valid_dataset))
X, y = next(iter(train_dataloader))
X.shape, y.shape


In [ ]:
class FrozenResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = resnet18(weights=WEIGHTS)
        for p in self.net.parameters():
            p.requires_grad = False
        # new modules are trainable by default
        self.net.fc = nn.Sequential(nn.Linear(self.net.fc.in_features, 1),
                                    nn.Sigmoid())

    def train(self, mode=True):
        super().train(mode)
        # requires_grad=False does not freeze BatchNorm: in train mode it would still
        # normalise with batch statistics and update its running averages
        for m in self.net.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
        return self

    def forward(self, x):
        return self.net(x)


model = FrozenResNet18().to(device).eval()
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"trainable parameters: {n_trainable} / {n_total}")

In [ ]:
hparams = dict(optimizer = 'Adam',
               lr = 1e-3,
               batch_size = batch_size,
               patience = 10,
               max_epochs = 50,
               loss = 'BCELoss',
               seed = SEED)

loss = nn.BCELoss()
optimizer = torch.optim.Adam(model.net.fc.parameters(), lr = hparams['lr'])

train_scores = []
valid_scores = []

CHECKPOINT = 'resnet18_checkpoint.pt'   # not 'checkpoint.pt', so notebook 04's checkpoint is kept
early_stopping = EarlyStopping(patience = hparams['patience'], verbose = False, path = CHECKPOINT)

## naive performance
print( "train accuracy:", get_model_accuracy(model, train_dataloader) )
print( "valid accuracy:", get_model_accuracy(model, valid_dataloader) )

In [ ]:
%%time

for t in range(hparams['max_epochs']):
    print('Epoch', len(train_scores) + 1)

    train_scores.append( train(train_dataloader,
                               model,
                               loss,
                               optimizer,
                               echo = True, echo_batch = False) )

    valid_scores.append( valid(valid_dataloader,
                               model,
                               loss,
                               echo = True) )

    early_stopping(valid_scores[-1], model)

    if early_stopping.early_stop:
        print("Early stopping")
        break

epochs_run = len(train_scores)
best_epoch = int(np.argmin(valid_scores)) + 1

# load the checkpoint with the best model
model.load_state_dict(torch.load(CHECKPOINT))
print(f"ran {epochs_run} epochs, best epoch {best_epoch}")

In [ ]:
plt.plot(train_scores, label = 'train')
plt.plot(valid_scores, label = 'validation')
plt.axvline(np.argmin(valid_scores), linestyle='--', color='r', label='Early Stopping Checkpoint')
plt.legend()
plt.xlabel('epoch')
plt.ylabel('BCE loss')
plt.show()

train_accuracy = float(get_model_accuracy(model, train_dataloader))
valid_accuracy = float(get_model_accuracy(model, valid_dataloader))
print( "train accuracy:", train_accuracy )
print( "valid accuracy:", valid_accuracy )

In [ ]:
# Final evaluation only: do not tune architecture, thresholds, or epochs on this score.
# Historical use of this folder for tuning means it is not a fresh external test.
test_dataset = torchvision.datasets.ImageFolder(
    TEST_DIR, loader=read_image_rgb, transform=rn_transform,
    target_transform=lambda x: torch.tensor([x], dtype=torch.float32))
assert test_dataset.class_to_idx == full_train_dataset.class_to_idx
test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)
test_accuracy = float(get_model_accuracy(model, test_dataloader))
print('test accuracy (historically exposed set):', test_accuracy)

MODEL_PATH = 'resnet18_pneumonia_model.pt'
META_PATH = 'resnet18_pneumonia_model.json'

metadata = {
    'model': {
        'architecture': 'resnet18',
        'pretrained_weights': 'ResNet18_Weights.IMAGENET1K_V1',
        'frozen_backbone': True,
        'head': 'fc = Sequential(Linear(512, 1), Sigmoid())',
        'output': 'probability of PNEUMONIA',
        'decision_threshold': 0.5,
        'trainable_parameters': n_trainable,
        'total_parameters': n_total,
        'state_dict_note': 'state dict of torchvision resnet18 with fc replaced by the head above (no wrapper prefix)',
    },
    'input': {
        'shape': [3, 224, 224],
        'color': 'RGB (read_image mode=ImageReadMode.RGB)',
        'scaling': 'ToDtype(float32, scale=True) -> [0, 1]',
        'normalize_mean': list(IMAGENET_MEAN),
        'normalize_std': list(IMAGENET_STD),
    },
    'data': {
        'class_to_idx': full_train_dataset.class_to_idx,
        'train_dir': TRAIN_DIR,
        'valid_source_dir': TRAIN_DIR,
        'test_dir': TEST_DIR,
        'split_seed': SEED,
        'validation_fraction': VALID_FRACTION,
        'split_method': 'class-stratified image-level split; patient independence unverified',
        'train_files': [str(Path(full_train_dataset.samples[i][0]).relative_to(TRAIN_DIR)) for i in train_indices],
        'valid_files': [str(Path(full_train_dataset.samples[i][0]).relative_to(TRAIN_DIR)) for i in valid_indices],
        'n_train': len(train_dataset),
        'n_valid': len(valid_dataset),
        'n_test': len(test_dataset),
        'note': 'Validation comes only from train/. Test is excluded from this training run but was used for tuning in earlier versions; not an untouched evaluation.',
    },
    'training': {
        **hparams,
        'epochs_run': epochs_run,
        'best_epoch': best_epoch,
        'best_valid_loss': float(min(valid_scores)),
        'train_losses': [float(x) for x in train_scores],
        'valid_losses': [float(x) for x in valid_scores],
    },
    'metrics': {
        'train_accuracy': train_accuracy,
        'valid_accuracy': valid_accuracy,
        'test_accuracy': test_accuracy,
    },
    'environment': {
        'torch': torch.__version__,
        'torchvision': torchvision.__version__,
        'device': device,
        'saved_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    },
}

torch.save({'model_state_dict': model.net.state_dict(),
            'train_accuracy': train_accuracy,
            'valid_accuracy': valid_accuracy,
        'test_accuracy': test_accuracy,
            'metadata': metadata},
           MODEL_PATH)
with open(META_PATH, 'w') as fh:
    json.dump(metadata, fh, indent=2)
print('saved', MODEL_PATH, 'and', META_PATH)

In [ ]:
## reload check: a plain torchvision resnet18 + the same head must give the same accuracy
ckpt = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)

reloaded = resnet18()
reloaded.fc = nn.Sequential(nn.Linear(reloaded.fc.in_features, 1), nn.Sigmoid())
reloaded.load_state_dict(ckpt['model_state_dict'], strict=True)
reloaded = reloaded.to(device).eval()

reloaded_accuracy = float(get_model_accuracy(reloaded, valid_dataloader))
assert np.isclose(reloaded_accuracy, ckpt['valid_accuracy']), (reloaded_accuracy, ckpt['valid_accuracy'])
print('reloaded valid accuracy:', reloaded_accuracy)